In [ ]:
# ============================================================
# CELL 0 — CONFIG (CIFAR BYZANTINE EXPERIMENT)
# ============================================================
DATASET        = "CIFAR10"
METHOD         = "ffum_median"   
SEED           = 17
SIGMA_SWEEP    = False           # Byzantine: only sigma=0 needed
SIGMA          = 0.0
BYZANTINE_FRAC = 0.2             # 20% of retain clients send poisoned gradients

# FL hyperparams (don't touch)
NUM_CLIENTS  = 10
DIRICHLET_A  = 0.1
BATCH        = 512
LOCAL_EPOCHS = 10
PRETRAIN_ROUNDS = {"MNIST": 80, "FashionMNIST": 50, "CIFAR10": 200}
UNLEARN_ROUNDS  = 20
POST_ROUNDS     = 5
MAX_EPOCHS   = 1
MIN_EPOCHS   = 1
ALPHA        = 0.5
GAMMA        = 1.0
CLIP_C       = 1.0
FORGET_CLIENT = 0
DELTA_DP     = 1e-5

# Cell 8 will just make sure no stale resume file is around.
RESTORE_CHECKPOINT = False

from pathlib import Path
OUT_DIR = Path("/kaggle/working/results")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[config] dataset={DATASET}  method={METHOD}  seed={SEED}")
print(f"[config] BYZANTINE_FRAC={BYZANTINE_FRAC}  sigma={SIGMA}")


In [ ]:
# ============================================================
# CELL 1 — IMPORTS & UTILITIES
# ============================================================
import os, math, json, time, copy, random, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[env] device={device}  cuda_name={torch.cuda.get_device_name(0) if device=='cuda' else 'N/A'}")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# ============================================================
# CELL 2 — MODEL DEFINITIONS
# ============================================================
class LeNet5(nn.Module):
    def __init__(self, in_ch=1, num_classes=10):
        super().__init__()
        self.c1 = nn.Conv2d(in_ch, 6, 5, padding=2)
        self.c2 = nn.Conv2d(6, 16, 5)
        self.f1 = nn.Linear(16*5*5, 120)
        self.f2 = nn.Linear(120, 84)
        self.f3 = nn.Linear(84, num_classes)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.c1(x)), 2)
        x = F.max_pool2d(F.relu(self.c2(x)), 2)
        x = x.flatten(1)
        x = F.relu(self.f1(x)); x = F.relu(self.f2(x))
        return self.f3(x)

class MiniLeNet(nn.Module):
    def __init__(self, in_ch=1, num_classes=10):
        super().__init__()
        self.c1 = nn.Conv2d(in_ch, 16, 3, padding=1)
        self.n1 = nn.GroupNorm(4, 16)
        self.c2 = nn.Conv2d(16, 32, 3, padding=1)
        self.n2 = nn.GroupNorm(4, 32)
        self.fc = nn.Linear(32*7*7, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.n1(self.c1(x))), 2)
        x = F.max_pool2d(F.relu(self.n2(self.c2(x))), 2)
        return self.fc(x.flatten(1))

def build_model(dataset):
    if dataset == "MNIST":        return LeNet5(1, 10)
    if dataset == "FashionMNIST": return MiniLeNet(1, 10)
    if dataset == "CIFAR10":
        m = models.resnet18(weights=None, num_classes=10)
        m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        m.maxpool = nn.Identity()
        return m

In [ ]:
# ============================================================
# CELL 3 — DATASET & DATA LOADERS
# ============================================================
def get_dataset(name):
    t_mnist = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    t_fm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,))
    ])
    t_cifar_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    ])
    t_cifar_val = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    ])
    root = "./data"
    if name == "MNIST":
        return (datasets.MNIST(root, True,  download=True, transform=t_mnist),
                datasets.MNIST(root, False, download=True, transform=t_mnist))
    if name == "FashionMNIST":
        return (datasets.FashionMNIST(root, True,  download=True, transform=t_fm),
                datasets.FashionMNIST(root, False, download=True, transform=t_fm))
    if name == "CIFAR10":
        return (datasets.CIFAR10(root, True,  download=True, transform=t_cifar_train),
                datasets.CIFAR10(root, False, download=True, transform=t_cifar_val))

def dirichlet_split(targets, n_clients, alpha, seed):
    rng = np.random.default_rng(seed)
    targets = np.asarray(targets)
    n_classes = int(targets.max() + 1)
    idx_by_class = [np.where(targets == c)[0] for c in range(n_classes)]
    for idx in idx_by_class: rng.shuffle(idx)
    client_idx = [[] for _ in range(n_clients)]
    for c in range(n_classes):
        props = rng.dirichlet([alpha] * n_clients)
        splits = (np.cumsum(props) * len(idx_by_class[c])).astype(int)[:-1]
        for i, chunk in enumerate(np.split(idx_by_class[c], splits)):
            client_idx[i].extend(chunk.tolist())
    return [np.array(c) for c in client_idx]

def make_loaders(train_ds, test_ds, seed):
    targets = train_ds.targets if hasattr(train_ds, "targets") else [y for _, y in train_ds]
    if torch.is_tensor(targets): targets = targets.numpy()
    parts = dirichlet_split(targets, NUM_CLIENTS, DIRICHLET_A, seed)
    client_loaders, client_sizes = [], []
    for idxs in parts:
        sub = Subset(train_ds, idxs.tolist())
        client_loaders.append(DataLoader(sub, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True))
        client_sizes.append(len(sub))
    val_loader    = DataLoader(test_ds, batch_size=1024, shuffle=False, num_workers=2)
    fsub          = Subset(train_ds, parts[FORGET_CLIENT].tolist())
    forget_loader = DataLoader(fsub, batch_size=256, shuffle=False, num_workers=2)
    return client_loaders, client_sizes, val_loader, forget_loader, parts

In [ ]:
# ============================================================
# CELL 4 — LOSS FUNCTIONS & DP UTILITIES
# ============================================================
def kl_div_loss(student_logits, teacher_logits):
    p = F.log_softmax(student_logits, -1)
    q = F.softmax(teacher_logits, -1)
    return F.kl_div(p, q, reduction="batchmean")

def js_div_loss(student_logits, teacher_logits):
    p = F.softmax(student_logits, -1)
    q = F.softmax(teacher_logits, -1)
    m = 0.5 * (p + q)
    return 0.5 * (F.kl_div(m.log(), p, reduction="batchmean") +
                  F.kl_div(m.log(), q, reduction="batchmean"))

def dp_clip_and_noise(model, sigma, clip_c=CLIP_C):
    """OUR contribution: per-layer clip + Gaussian noise. sigma=0 → no-op."""
    if sigma <= 0: return
    with torch.no_grad():
        for p in model.parameters():
            if p.grad is None: continue
            g = p.grad
            norm = g.norm(2)
            g.mul_(min(1.0, clip_c / (norm + 1e-12)))
            std = sigma * clip_c / math.sqrt(max(g.numel(), 1))
            g.add_(torch.randn_like(g) * std)

def compute_epsilon(sigma, steps, q, delta=DELTA_DP):
    if sigma <= 0 or steps == 0: return float("inf")
    best = float("inf")
    for alpha in [1.25, 1.5, 1.75, 2, 2.5, 3, 4, 8, 16, 32, 64, 128]:
        rdp = steps * q * q * alpha / (2 * sigma * sigma)
        eps = rdp + math.log(1/delta) / (alpha - 1)
        best = min(best, eps)
    return best

In [ ]:
# ============================================================
# CELL 5 — FEDERATED TRAINING UTILITIES
# ============================================================
def local_train_ce(model, loader, epochs, lr):
    if DATASET == "FashionMNIST":
        mom = 0.0; wd = 0.0
    elif DATASET == "CIFAR10":
        mom = 0.9; wd = 5e-4
    else:  # MNIST
        mom = 0.9; wd = 0.0

    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=mom, weight_decay=wd)

    total_steps = epochs * len(loader)
    if DATASET == "CIFAR10" and total_steps > 1:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps, eta_min=lr * 0.01)
    else:
        scheduler = None

    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            F.cross_entropy(model(x), y).backward()
            opt.step()
            if scheduler is not None:
                scheduler.step()
    return model

def fed_avg(state_dicts, weights):
    w = np.array(weights, dtype=np.float64); w = w / w.sum()
    out = copy.deepcopy(state_dicts[0])
    for k in out:
        out[k] = sum(float(w[i]) * state_dicts[i][k].float() for i in range(len(state_dicts)))
    return out

def coord_median(state_dicts):
    out = copy.deepcopy(state_dicts[0])
    for k in out:
        stk = torch.stack([sd[k].float() for sd in state_dicts], dim=0)
        out[k] = stk.median(dim=0).values
    return out

def evaluate(model, loader):
    model.eval(); correct = tot = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item(); tot += y.numel()
    return correct / tot

def pretrain(client_loaders, client_sizes, val_loader, rounds, resume_path=None):
    global_m = build_model(DATASET).to(device)
    start_round = 0
    if resume_path is not None and resume_path.exists():
        ckpt = torch.load(resume_path, map_location=device, weights_only=False)
        global_m.load_state_dict(ckpt["model"])
        start_round = ckpt["round"]
        print(f"  [resume] loaded round {start_round}, val_acc={evaluate(global_m, val_loader):.4f}")

    for r in range(start_round, rounds):
        if DATASET == "CIFAR10":
            if r < 100:   lr_pre = 0.10
            elif r < 150: lr_pre = 0.02
            else:         lr_pre = 0.004
        else:
            lr_pre = {"MNIST": 0.05, "FashionMNIST": 0.05}[DATASET]

        sds, sizes = [], []
        for c, loader in enumerate(client_loaders):
            local = copy.deepcopy(global_m)
            local_train_ce(local, loader, epochs=LOCAL_EPOCHS, lr=lr_pre)
            sds.append(local.state_dict()); sizes.append(client_sizes[c])
        global_m.load_state_dict(fed_avg(sds, sizes))

        if (r+1) % 10 == 0 or r == rounds-1:
            acc = evaluate(global_m, val_loader)
            print(f"  [pretrain round {r+1:03d}/{rounds}]  lr={lr_pre:.4f}  val_acc={acc:.4f}")
            if resume_path is not None:
                torch.save({"model": global_m.state_dict(), "round": r+1}, resume_path)
                print(f"  [ckpt saved] round {r+1}")
    return global_m

In [ ]:
# ============================================================
# CELL 6 — UNLEARNING STEPS
# ============================================================
def ffum_max_step(model, teacher, forget_loader, sigma):
    """Maximize KL on forget data → gradient ASCENT, capped at KL > 5."""
    lr_max = {"MNIST": 0.005, "FashionMNIST": 0.002, "CIFAR10": 0.01}[DATASET]
    opt = torch.optim.SGD(model.parameters(), lr=lr_max)
    model.train(); teacher.eval()
    for _ in range(MAX_EPOCHS):
        for x, _ in forget_loader:
            x = x.to(device)
            opt.zero_grad()
            with torch.no_grad(): t_logits = teacher(x)
            kl = kl_div_loss(model(x), t_logits)
            if kl.item() > 5.0:
                continue
            loss = -kl
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            dp_clip_and_noise(model, sigma)
            opt.step()
    return model

def ffum_min_step(model, teacher, retain_loader, sigma):
    """Minimize α·JS + γ·CE on retain data."""
    lr_min = {"MNIST": 0.02, "FashionMNIST": 0.05, "CIFAR10": 0.02}[DATASET]
    mom_min = 0.9 if DATASET == "CIFAR10" else 0.0
    opt = torch.optim.SGD(model.parameters(), lr=lr_min, momentum=mom_min)
    model.train(); teacher.eval()
    for _ in range(MIN_EPOCHS):
        for x, y in retain_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.no_grad(): t_logits = teacher(x)
            s_logits = model(x)
            loss = ALPHA * js_div_loss(s_logits, t_logits) + GAMMA * F.cross_entropy(s_logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            dp_clip_and_noise(model, sigma)
            opt.step()
    return model

def poison_state_dict(sd):
    out = copy.deepcopy(sd)
    for k in out:
        if out[k].is_floating_point():
            out[k] = out[k] * -5.0
    return out

def run_unlearn(pretrained, client_loaders, client_sizes, val_loader, forget_loader, sigma, method):
    teacher = copy.deepcopy(pretrained).to(device).eval()
    for p in teacher.parameters(): p.requires_grad_(False)
    global_m = copy.deepcopy(pretrained).to(device)
    n_byz = int(round(BYZANTINE_FRAC * (NUM_CLIENTS - 1)))

    for t in range(UNLEARN_ROUNDS):
        local_f = copy.deepcopy(global_m)
        ffum_max_step(local_f, teacher, forget_loader, sigma)
        global_m.load_state_dict(local_f.state_dict())
        sds, sizes = [], []
        for c in range(NUM_CLIENTS):
            if c == FORGET_CLIENT: continue
            local_c = copy.deepcopy(global_m)
            ffum_min_step(local_c, teacher, client_loaders[c], sigma)
            sd = local_c.state_dict()
            if n_byz > 0 and (c - 1) < n_byz:
                sd = poison_state_dict(sd)
            sds.append(sd); sizes.append(client_sizes[c])
        global_m.load_state_dict(coord_median(sds) if method == "ffum_median" else fed_avg(sds, sizes))

    for _ in range(POST_ROUNDS):
        sds, sizes = [], []
        for c in range(NUM_CLIENTS):
            if c == FORGET_CLIENT: continue
            local_c = copy.deepcopy(global_m)
            ffum_min_step(local_c, teacher, client_loaders[c], sigma)
            sds.append(local_c.state_dict()); sizes.append(client_sizes[c])
        global_m.load_state_dict(coord_median(sds) if method == "ffum_median" else fed_avg(sds, sizes))
    return global_m

In [ ]:
# ============================================================
# CELL 7 — EVALUATION: MIA + MODEL DISTANCE (class-balanced)
# ============================================================
def per_sample_loss(model, loader):
    model.eval(); out = []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0).clamp(-50.0, 50.0)
            loss = F.cross_entropy(logits, y, reduction="none")
            loss = torch.nan_to_num(loss, nan=50.0, posinf=50.0, neginf=0.0)
            out.append(loss.cpu().numpy())
    arr = np.concatenate(out)
    return np.nan_to_num(arr, nan=50.0, posinf=50.0, neginf=0.0)

def compute_mia(model, forget_loader, val_loader):
    def collect(loader):
        model.eval(); L, Y = [], []
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                logits = torch.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0).clamp(-50.0, 50.0)
                loss = F.cross_entropy(logits, y, reduction="none")
                loss = torch.nan_to_num(loss, nan=50.0, posinf=50.0, neginf=0.0)
                L.append(loss.cpu().numpy()); Y.append(y.cpu().numpy())
        return np.concatenate(L), np.concatenate(Y)

    l_f, y_f = collect(forget_loader)
    l_v, y_v = collect(val_loader)

    # Symmetric per-class balancing: take min(n_forget_c, n_val_c) from each class
    rng = np.random.default_rng(0)
    f_idx, v_idx = [], []
    for c in np.unique(np.concatenate([y_f, y_v])):
        pool_f = np.where(y_f == c)[0]
        pool_v = np.where(y_v == c)[0]
        n_c = min(len(pool_f), len(pool_v))
        if n_c == 0: continue
        f_idx.append(rng.choice(pool_f, size=n_c, replace=False))
        v_idx.append(rng.choice(pool_v, size=n_c, replace=False))
    f_idx = np.concatenate(f_idx)
    v_idx = np.concatenate(v_idx)
    l_f = l_f[f_idx]
    l_v = l_v[v_idx]

    # 5 bootstrap resamples × 5-fold CV = 25 estimates, averaged
    accs_all = []
    for boot_seed in range(5):
        rng_boot = np.random.default_rng(boot_seed)
        n = min(len(l_f), len(l_v))
        idx_f = rng_boot.choice(len(l_f), size=n, replace=False)
        idx_v = rng_boot.choice(len(l_v), size=n, replace=False)
        X = np.concatenate([l_f[idx_f], l_v[idx_v]]).reshape(-1, 1)
        y = np.concatenate([np.ones(n), np.zeros(n)])
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=boot_seed)
        for tr, te in skf.split(X, y):
            clf = LogisticRegression(max_iter=1000).fit(X[tr], y[tr])
            accs_all.append(clf.score(X[te], y[te]))
    return float(np.mean(accs_all))

def model_l2_distance(model_a, model_b):
    sa = model_a.state_dict(); sb = model_b.state_dict()
    total = 0.0
    for k in sa:
        if sa[k].is_floating_point():
            total += float(((sa[k] - sb[k]) ** 2).sum().item())
    return float(total ** 0.5)

In [ ]:
# ============================================================
# CELL 8 — COPY FINAL PRETRAINED CHECKPOINT FROM DATASET
# For Byzantine experiment: we have the FINAL pretrain_CIFAR10_seed17.pt
# (~87% model). Copy it to OUT_DIR so Cell 9 loads it instead of retraining.
# Update src path below to match your Kaggle dataset input location.
# ============================================================
import os, shutil, zipfile
from pathlib import Path

dst_dir = Path("/kaggle/working/results")
dst_dir.mkdir(parents=True, exist_ok=True)
final_name = f"pretrain_{DATASET}_seed{SEED}.pt"
dst = dst_dir / final_name

# Find the .pt file anywhere in /kaggle/input/
candidates = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == final_name or (final_name.replace(".pt", "") in f and f.endswith(".pt")):
            candidates.append(Path(root) / f)

if candidates:
    src = candidates[0]
    print(f"[restore] found: {src}")
    if src.is_file():
        shutil.copy(src, dst)
        print(f"[restore] copied → {dst}  ({dst.stat().st_size/1e6:.1f} MB)")
else:
    # Fall back: look for extracted directory (Kaggle auto-extracted a .pt)
    extracted_dirs = []
    for root, dirs, files in os.walk("/kaggle/input"):
        for d in dirs:
            if d == final_name.replace(".pt", "") or d == final_name:
                extracted_dirs.append(Path(root) / d)
    if extracted_dirs:
        src_dir = extracted_dirs[0]
        print(f"[restore] found extracted dir: {src_dir}")
        archive_name = final_name.replace(".pt", "")
        with zipfile.ZipFile(dst, "w", zipfile.ZIP_STORED) as zf:
            for root, dirs, files in os.walk(src_dir):
                for f in files:
                    fp = os.path.join(root, f)
                    rel = os.path.relpath(fp, src_dir)
                    zf.write(fp, os.path.join(archive_name, rel))
        print(f"[restore] repacked → {dst}  ({dst.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"[restore] ERROR: could not find {final_name} in /kaggle/input/")
        print("Listing /kaggle/input/:")
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                print(" ", os.path.join(root, f))

# Also delete any stale .resume.pt (not needed for Byzantine)
resume_path = dst_dir / f"pretrain_{DATASET}_seed{SEED}.resume.pt"
if resume_path.exists():
    resume_path.unlink()
    print(f"[restore] cleaned stale resume file")


In [ ]:
# ============================================================
# CELL 9 — PRETRAIN
# ============================================================
set_seed(SEED)
train_ds, test_ds = get_dataset(DATASET)
client_loaders, client_sizes, val_loader, forget_loader, parts = make_loaders(train_ds, test_ds, SEED)

ckpt_path   = OUT_DIR / f"pretrain_{DATASET}_seed{SEED}.pt"
resume_path = OUT_DIR / f"pretrain_{DATASET}_seed{SEED}.resume.pt"

if ckpt_path.exists():
    print(f"[pretrain] loading FINAL cached checkpoint: {ckpt_path}")
    pretrained = build_model(DATASET).to(device)
    pretrained.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False))
    pre_acc = evaluate(pretrained, val_loader)
    print(f"[pretrain] loaded  acc={pre_acc:.4f}")
else:
    print("[pretrain] starting..." + (" (resuming from checkpoint)" if resume_path.exists() else " (from scratch)"))
    t0 = time.time()
    pretrained = pretrain(client_loaders, client_sizes, val_loader,
                          PRETRAIN_ROUNDS[DATASET], resume_path=resume_path)
    pre_acc = evaluate(pretrained, val_loader)
    print(f"[pretrain] done in {time.time()-t0:.0f}s  acc={pre_acc:.4f}")
    torch.save(pretrained.state_dict(), ckpt_path)
    print(f"[pretrain] saved final → {ckpt_path}")
    if resume_path.exists(): resume_path.unlink()

mia_before = compute_mia(pretrained, forget_loader, val_loader)
print(f"[mia_before] {mia_before:.4f}")
print(f"\n>>> Pretrain acc = {pre_acc*100:.2f}%  "
      f"(target: {'~78%' if DATASET=='CIFAR10' else '~99%' if DATASET=='MNIST' else '~81%'})")

In [ ]:
# ============================================================
# CELL 10 — MANUAL CHECKPOINT SAVE
# Run this cell manually at any time while Cell 9 is still
# running (open a second tab or interrupt briefly) if you
# want to grab the latest .resume.pt before the session dies.
# The file will appear in the Kaggle output tab for download.
# Upload it to a Kaggle dataset, then set
# RESTORE_CHECKPOINT = True in Cell 0 to resume next session.
# ============================================================
resume_path_check = OUT_DIR / f"pretrain_{DATASET}_seed{SEED}.resume.pt"
visible_copy      = Path("/kaggle/working") / f"pretrain_{DATASET}_seed{SEED}.resume.pt"

if resume_path_check.exists():
    shutil.copy(resume_path_check, visible_copy)
    print(f"[manual save] Copied to Kaggle output tab: {visible_copy.name}")
    print("Download it now from the output panel on the right.")
    print("Then upload to a Kaggle dataset and set RESTORE_CHECKPOINT=True in Cell 0.")
else:
    print("[manual save] No .resume.pt found yet.")
    print("Either pretraining hasn't started, or it already finished and saved the final .pt.")

In [ ]:
# ============================================================
# CELL 11 — UNLEARN + SIGMA SWEEP (Byzantine-aware filenames)
# ============================================================
results = []
sigmas = [0.0, 0.5, 1.0, 2.0, 4.0] if SIGMA_SWEEP else [SIGMA]

for sig in sigmas:
    print(f"\n[unlearn] σ={sig}  method={METHOD}  byzantine={BYZANTINE_FRAC}")
    set_seed(SEED)
    t0 = time.time()
    unlearned = run_unlearn(pretrained, client_loaders, client_sizes, val_loader, forget_loader, sig, METHOD)
    un_acc = evaluate(unlearned, val_loader)
    mia_after = compute_mia(unlearned, forget_loader, val_loader)
    l2_dist  = model_l2_distance(unlearned, pretrained)
    n_forget = client_sizes[FORGET_CLIENT]
    n_total  = sum(client_sizes)
    eps = compute_epsilon(sig, UNLEARN_ROUNDS + POST_ROUNDS, n_forget / n_total)
    print(f"[done] {time.time()-t0:.0f}s  pre={pre_acc:.4f}  un={un_acc:.4f}  "
          f"MIA={mia_after:.4f}  L2={l2_dist:.3f}  ε={eps:.3f}")
    row = dict(dataset=DATASET, method=METHOD, sigma=sig, seed=SEED,
               byzantine_frac=BYZANTINE_FRAC,
               pretrain_acc=pre_acc, unlearn_acc=un_acc,
               mia_before=mia_before, mia_after=mia_after,
               l2_distance=l2_dist, epsilon=eps,
               n_forget=n_forget, n_total=n_total)
    results.append(row)
    byz_tag = f"_byz{BYZANTINE_FRAC}" if BYZANTINE_FRAC > 0 else ""
    out_csv = OUT_DIR / f"results_{DATASET}_{METHOD}_sig{sig}_seed{SEED}{byz_tag}.csv"
    with open(out_csv, "w") as f:
        f.write(",".join(row.keys()) + "\n")
        f.write(",".join(str(v) for v in row.values()) + "\n")
    print(f"[save] {out_csv}")

print("\nDONE")
for r in results:
    print(r)


In [ ]:
# ============================================================
# CELL 12 — Aggregate across seeds → mean±std for paper Table 1
# Run after you have results CSVs for all three seeds.
# ============================================================
import glob, csv
from collections import defaultdict

rows = []
for fp in sorted(glob.glob(str(OUT_DIR / "results_*.csv"))):
    if "ALL" in fp: continue
    with open(fp) as f:
        for r in csv.DictReader(f):
            rows.append(r)

print(f"[agg] loaded {len(rows)} runs")

groups = defaultdict(list)
for r in rows:
    key = (r["dataset"], r["method"], float(r["sigma"]))
    groups[key].append(r)

def stat(rs, k):
    vs = [float(r[k]) for r in rs if r.get(k) not in (None, "", "inf")]
    if not vs: return (float("nan"), float("nan"))
    return (float(np.mean(vs)), float(np.std(vs)))

print("\n=== TABLE 1 (paste into LaTeX) ===")
print("Dataset & Method & σ & Pretrain & Unlearn & MIA & L2 & ε \\\\")
print("\\hline")
for (ds, m, sig), rs in sorted(groups.items()):
    pa, _    = stat(rs, "pretrain_acc")
    ua, ua_s = stat(rs, "unlearn_acc")
    ma, ma_s = stat(rs, "mia_after")
    la, la_s = stat(rs, "l2_distance")
    ev = [float(r["epsilon"]) for r in rs if r["epsilon"] not in ("inf", "")]
    eps_str = f"{np.mean(ev):.2f}" if ev else "$\\infty$"
    n = len(rs)
    print(f"{ds} & {m} & {sig} & "
          f"{pa*100:.2f} & "
          f"{ua*100:.2f}$\\pm${ua_s*100:.2f} & "
          f"{ma*100:.2f}$\\pm${ma_s*100:.2f} & "
          f"{la:.2f}$\\pm${la_s:.2f} & "
          f"{eps_str} \\\\  % n_seeds={n}")

In [ ]:
# ============================================================
# CELL 13 — Figure 1: 3-panel privacy-utility tradeoff
# ============================================================
import glob, csv, os
from collections import defaultdict
import matplotlib.pyplot as plt
os.makedirs("figs", exist_ok=True)

rows = []
for fp in sorted(glob.glob(str(OUT_DIR / "results_*.csv"))):
    if "ALL" in fp: continue
    with open(fp) as f:
        for r in csv.DictReader(f):
            rows.append(r)

PLOT_DATASET = DATASET
PLOT_METHOD  = "ffum"

g = defaultdict(list)
for r in rows:
    if r["dataset"] == PLOT_DATASET and r["method"] == PLOT_METHOD:
        g[float(r["sigma"])].append(r)

sigmas = sorted(g.keys())
def msd(rs, k):
    v = [float(r[k]) for r in rs if r.get(k) not in (None, "", "inf")]
    return (np.mean(v), np.std(v)) if v else (np.nan, 0)

acc_m = [msd(g[s], "unlearn_acc")[0]*100 for s in sigmas]
acc_s = [msd(g[s], "unlearn_acc")[1]*100 for s in sigmas]
mia_m = [msd(g[s], "mia_after")[0]*100   for s in sigmas]
mia_s = [msd(g[s], "mia_after")[1]*100   for s in sigmas]
eps_m = [np.mean([float(r["epsilon"]) for r in g[s] if r["epsilon"] != "inf"])
         if any(r["epsilon"] != "inf" for r in g[s]) else np.inf for s in sigmas]

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
ax[0].errorbar(sigmas, acc_m, yerr=acc_s, marker="o", capsize=3, color="tab:blue")
ax[0].set_xlabel("DP noise σ"); ax[0].set_ylabel("Unlearn accuracy (%)")
ax[0].set_title("(a) Utility vs σ"); ax[0].grid(alpha=.3)

ax[1].errorbar(sigmas, mia_m, yerr=mia_s, marker="s", capsize=3, color="tab:red")
ax[1].axhline(50, ls="--", color="k", lw=1, label="ideal (50%)")
ax[1].set_xlabel("DP noise σ"); ax[1].set_ylabel("MIA score (%)")
ax[1].set_title("(b) Unlearning quality vs σ"); ax[1].legend(); ax[1].grid(alpha=.3)

pareto = [(e, a, sd) for e, a, sd, s in zip(eps_m, acc_m, acc_s, sigmas) if s > 0 and np.isfinite(e)]
if pareto:
    e_, a_, s_ = zip(*pareto)
    ax[2].errorbar(e_, a_, yerr=s_, marker="^", capsize=3, color="tab:green")
    ax[2].set_xscale("log")
    ax[2].set_xlabel("Privacy budget ε (log scale)")
    ax[2].set_ylabel("Unlearn accuracy (%)")
    ax[2].set_title("(c) Privacy–utility Pareto"); ax[2].grid(alpha=.3)

plt.suptitle(f"DP-f-FUM tradeoff on {PLOT_DATASET}", y=1.02)
plt.tight_layout()
plt.savefig("figs/fig_tradeoff.pdf", bbox_inches="tight")
plt.savefig("figs/fig_tradeoff.png", bbox_inches="tight", dpi=150)
plt.show()
print("[save] figs/fig_tradeoff.pdf  ←  paste into LaTeX")